# Multi-task ablation on Kaggle (one share of the runs per notebook)

Full guide: `docs/kaggle_ablation.md` in the repository.

**Before running** (right-hand panel):
1. *Settings > Accelerator*: **GPU T4 x2** (the code uses one GPU).
2. *Settings > Internet*: **on**.
3. *Add-ons > Secrets*: secret **`HF_TOKEN`** (Hugging Face read token of the account that accepted the FiQA access form), ticked for **this** notebook.

Then edit **only the first code cell** (which seeds / variants this notebook runs) and use **Save Version > Save & Run All (Commit)**.

In [ ]:
# ---- the only cell to edit -------------------------------------------------------
import os
os.environ["SEEDS"] = "42"                                   # this notebook's share, e.g. "42", "43", "44"
os.environ["VARIANTS"] = "base hier sent conf ord mark all"  # or a subset, e.g. "base mark"
os.environ["EPOCHS"] = "10"                                  # 20 if the seed study says longer training helps
os.environ["MODEL"] = "roberta-base"
BRANCH = "claude/adoring-fermat-r9ta0k"
# ------------------------------------------------------------------------------------
print({k: os.environ[k] for k in ("SEEDS", "VARIANTS", "EPOCHS", "MODEL")})

In [ ]:
%cd /kaggle/working
!git clone -q https://github.com/ptrnghieu/SEntFiN
%cd SEntFiN
!git checkout -q {BRANCH} && git log --oneline -1

In [ ]:
# torch is preinstalled on Kaggle; this adds the data-pipeline deps and pytorch-crf
!pip install -q -r requirements.txt pytorch-crf
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), "No GPU: set Settings > Accelerator to GPU T4"


In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')  # read by huggingface_hub
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [ ]:
# Rebuild data/processed (~15 s): downloads FiQA (gated) and FinEntity; LLM labels are in the repo
!python -m absa_data build --require-fiqa

## Ablation runs
Results: `runs/ablation/multitask/<variant>-s<seed>/`; this notebook's mean ± sd: `runs/ablation/SEEDS.md`. Checkpoints are deleted after each run.

In [ ]:
%%bash
set -euo pipefail
bash scripts/run_ablation.sh 2>&1 | grep -v -i warning

In [ ]:
from pathlib import Path
from IPython.display import Markdown
p = Path('runs/ablation/SEEDS.md')
Markdown(p.read_text() if p.exists() else 'no runs/ablation/SEEDS.md yet')

In [ ]:
!du -sh runs; find runs/ablation -name metrics.json | sort